# 04. Model Comparison


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import sys

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.models import MLP, GRU, TCN, MODEL_REGISTRY
from src.preprocessing import load_cached_features, get_grouped_splits, create_dataloader, compute_class_weights
from src.train import train_model, run_cross_validation, get_training_device
from src.evaluate import predict, evaluate_predictions, analyze_views, plot_and_save_confusion_matrix

OUT = ROOT / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
dev = get_training_device()
print(f"Training device: {dev}")

Training device: cuda


## 1. Load Data

In [2]:
cache_path = ROOT / "data" / "processed" / "squat_pose_16f_v1.npz"
data = load_cached_features(cache_path)
X, y, groups, views, paths = data["X"], data["y"], data["groups"], data["views"], data["paths"]

train_idx, val_idx, test_idx = get_grouped_splits(X, y, groups, seed=42)
train_loader = create_dataloader(X, y, train_idx, batch_size=32, shuffle=True)
val_loader   = create_dataloader(X, y, val_idx, batch_size=32, shuffle=False)
test_loader  = create_dataloader(X, y, test_idx, batch_size=32, shuffle=False)

class_weights = compute_class_weights(y[train_idx], device=dev)
print(f"Train clips: {len(train_idx)} | Val clips: {len(val_idx)} | Test clips: {len(test_idx)}")

Train clips: 90 | Val clips: 30 | Test clips: 30


## 2. Train Models

In [3]:
trained_models = {}
for name, Cls in MODEL_REGISTRY.items():
    print(f"\nTraining {name}...")
    m, val_f1 = train_model(Cls(), train_loader, val_loader, class_weights=class_weights, device=dev)
    trained_models[name] = m
    print(f"  {name} Best Val Macro-F1: {val_f1:.4f}")


Training MLP...


  Early stop at epoch 24
  MLP Best Val Macro-F1: 0.5333

Training GRU...


  Early stop at epoch 15
  GRU Best Val Macro-F1: 0.6122

Training TCN...


  Early stop at epoch 31
  TCN Best Val Macro-F1: 0.5982


## 3. Held-Out Evaluation

In [4]:
results = []
for name, model in trained_models.items():
    y_true, y_pred, y_prob = predict(model, test_loader, device=dev)
    metrics = evaluate_predictions(y_true, y_pred)
    acc = (y_true == y_pred).mean()
    results.append({
        "Model": name,
        "Accuracy": round(acc, 4),
        "Macro-F1": round(metrics["macro_f1"], 4),
        "Balanced Accuracy": round(metrics["balanced_accuracy"], 4),
    })
    print(f"\n=== {name} Classification Report ===")
    print(metrics["report_str"])

# Add Majority Baseline
maj_pred = np.full(len(test_idx), np.bincount(y[train_idx]).argmax())
results.append({
    "Model": "Majority Baseline",
    "Accuracy": round((y[test_idx] == maj_pred).mean(), 4),
    "Macro-F1": round(evaluate_predictions(y[test_idx], maj_pred)["macro_f1"], 4),
    "Balanced Accuracy": round(evaluate_predictions(y[test_idx], maj_pred)["balanced_accuracy"], 4),
})

res_df = pd.DataFrame(results).sort_values("Macro-F1", ascending=False)
display(res_df)


=== MLP Classification Report ===
              precision    recall  f1-score   support

         bad       0.50      0.60      0.55        15
        good       0.50      0.40      0.44        15

    accuracy                           0.50        30
   macro avg       0.50      0.50      0.49        30
weighted avg       0.50      0.50      0.49        30


=== GRU Classification Report ===
              precision    recall  f1-score   support

         bad       0.50      0.60      0.55        15
        good       0.50      0.40      0.44        15

    accuracy                           0.50        30
   macro avg       0.50      0.50      0.49        30
weighted avg       0.50      0.50      0.49        30




=== TCN Classification Report ===
              precision    recall  f1-score   support

         bad       0.62      0.87      0.72        15
        good       0.78      0.47      0.58        15

    accuracy                           0.67        30
   macro avg       0.70      0.67      0.65        30
weighted avg       0.70      0.67      0.65        30



,Model,Accuracy,Macro-F1,Balanced Accuracy
2,TCN,0.6667,0.6528,0.6667
0,MLP,0.5000,0.4949,0.5000
1,GRU,0.5000,0.4949,0.5000
3,Majority Baseline,0.5000,0.3333,0.5000


## 4. Grouped 5-Fold Cross-Validation

In [5]:
cv_summary = run_cross_validation(cache_path=cache_path, n_folds=5, out_dir=OUT)
display(cv_summary[["Macro-F1", "Bal Acc"]].sort_values("Macro-F1", ascending=False))

Starting 5-Fold Grouped CV on 25 subjects (16 frames per clip)...

  Fold 1/5 | Test subjects: [np.str_('subject_005'), np.str_('subject_010'), np.str_('subject_015'), np.str_('subject_020'), np.str_('subject_026')]


  MLP   | Acc=0.467 | Macro-F1=0.426 | Bal-Acc=0.467 | val-F1=0.374


  GRU   | Acc=0.433 | Macro-F1=0.417 | Bal-Acc=0.433 | val-F1=0.541


  TCN   | Acc=0.500 | Macro-F1=0.486 | Bal-Acc=0.500 | val-F1=0.583

  Fold 2/5 | Test subjects: [np.str_('subject_004'), np.str_('subject_009'), np.str_('subject_014'), np.str_('subject_019'), np.str_('subject_024')]


  MLP   | Acc=0.467 | Macro-F1=0.444 | Bal-Acc=0.467 | val-F1=0.580


  GRU   | Acc=0.533 | Macro-F1=0.531 | Bal-Acc=0.533 | val-F1=0.624


... (8 outputs left out)

  GRU   | Acc=0.533 | Macro-F1=0.531 | Bal-Acc=0.533 | val-F1=0.571


  TCN   | Acc=0.667 | Macro-F1=0.661 | Bal-Acc=0.667 | val-F1=0.657

[RESULTS] Grouped 5-Fold Cross-Validation Summary (16f):


               Accuracy       Macro-F1        Bal Acc
model                                                
TCN       0.567 ± 0.071  0.552 ± 0.069  0.567 ± 0.071
GRU       0.507 ± 0.043  0.478 ± 0.071  0.507 ± 0.043
MLP       0.453 ± 0.038  0.438 ± 0.036  0.453 ± 0.038
Majority  0.500 ± 0.000  0.333 ± 0.000  0.500 ± 0.000

Saved CV metrics to D:\DL\outputs/cv_summary_16f.csv


,Macro-F1,Bal Acc
model,,
TCN,0.552 ± 0.069,0.567 ± 0.071
GRU,0.478 ± 0.071,0.507 ± 0.043
MLP,0.438 ± 0.036,0.453 ± 0.038
Majority,0.333 ± 0.000,0.500 ± 0.000


## 5. View-Wise Breakdown

In [6]:
best_model = trained_models["GRU"]
y_true, y_pred, _ = predict(best_model, test_loader, device=dev)
view_metrics = analyze_views(y_true, y_pred, views[test_idx])
display(view_metrics)

plot_and_save_confusion_matrix(y_true, y_pred, OUT / "gru_confusion_matrix.png", "GRU Test Confusion Matrix")

,view,clips,macro_f1,bal_accuracy
0,diagonal,10,0.4000,0.4
1,front,10,0.6970,0.7
2,side,10,0.2857,0.4


Confusion matrix plot saved to D:\DL\outputs\gru_confusion_matrix.png
